# Member 2 — Heuristic CVRP Methods | Version 2: Simulated Annealing
**Git commit:** `feat(heuristic): add simulated annealing route improvement`

Starts from Clarke–Wright routes and searches for shorter feasible solutions using relocate, swap, and intra-route 2-opt moves. Infeasible candidates are rejected. The algorithm is stochastic, so record the random seed and parameters.

## 1. Dependencies and shared instance

In [ ]:
!pip -q install numpy pandas matplotlib

In [ ]:
import math, random, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
N_CUSTOMERS = 20   # Increase to 31 to match the scale of the sample report
K = 5
CAPACITY = 100
rng = np.random.default_rng(SEED)

# Shared deterministic synthetic instance. Both members must use these same parameters.
coords = [(50.0, 50.0)] + [(float(x), float(y)) for x,y in rng.uniform(0,100,size=(N_CUSTOMERS,2))]
demands = [0] + [int(x) for x in rng.integers(4, 11, size=N_CUSTOMERS)]
n = len(coords)
dist = np.zeros((n,n), dtype=int)
for i in range(n):
    for j in range(n):
        if i != j:
            dist[i,j] = int(math.floor(math.dist(coords[i], coords[j]) + 0.5))

print(f"Nodes: {n} (including depot 0); customers: {N_CUSTOMERS}; vehicles: {K}; capacity: {CAPACITY}")
print(f"Total demand: {sum(demands)}; total fleet capacity: {K*CAPACITY}")
assert sum(demands) <= K*CAPACITY, "Infeasible generated instance; adjust seed/demands."

In [ ]:
def route_distance(route):
    """Distance for a customer list; depot 0 is added at both ends."""
    seq = [0] + list(route) + [0]
    return int(sum(dist[seq[i], seq[i+1]] for i in range(len(seq)-1)))

def solution_cost(routes):
    return sum(route_distance(r) for r in routes)

def validate_routes(routes, verbose=True):
    errors = []
    seen = [c for route in routes for c in route]
    if sorted(seen) != list(range(1,n)):
        errors.append("Every customer must appear exactly once.")
    if len(routes) > K:
        errors.append(f"Used {len(routes)} routes, exceeding fleet limit {K}.")
    for ridx, route in enumerate(routes, start=1):
        load = sum(demands[c] for c in route)
        if load > CAPACITY:
            errors.append(f"Route {ridx} load {load} exceeds capacity {CAPACITY}.")
        if not route:
            errors.append(f"Route {ridx} is empty.")
    cost = solution_cost(routes)
    if verbose:
        print("Feasible:", not errors, "| Cost:", cost, "| Routes:", len(routes))
        for e in errors: print(" -", e)
    return {"feasible": not errors, "cost": cost, "routes_used": len(routes), "errors": errors}

## 2. Clarke–Wright initial solution

In [ ]:
def clarke_wright_savings():
    routes={i:[i] for i in range(1,n)}
    route_of={i:i for i in range(1,n)}
    savings=sorted([(int(dist[0,i]+dist[0,j]-dist[i,j]),i,j)
                    for i in range(1,n) for j in range(i+1,n)], reverse=True)
    for saving,i,j in savings:
        ri,rj=route_of.get(i),route_of.get(j)
        if ri is None or rj is None or ri==rj: continue
        a,b=routes[ri][:],routes[rj][:]
        if a[-1]!=i:
            if a[0]==i: a.reverse()
            else: continue
        if b[0]!=j:
            if b[-1]==j: b.reverse()
            else: continue
        if sum(demands[c] for c in a+b)>CAPACITY: continue
        routes[ri]=a+b; del routes[rj]
        for c in routes[ri]: route_of[c]=ri
    return [r for r in routes.values() if r]

initial_routes=clarke_wright_savings()
print("Initial solution:", validate_routes(initial_routes))

## 3. Feasible neighbourhood moves and Simulated Annealing

In [ ]:
def route_load(route): return sum(demands[c] for c in route)

def make_neighbor(routes, rng):
    """Return a feasible neighbouring solution or None if no valid move was found."""
    current=[r[:] for r in routes]
    for _ in range(30):
        cand=[r[:] for r in current]
        move=rng.choice(["relocate","swap","two_opt"])
        nonempty=[i for i,r in enumerate(cand) if r]
        if move=="relocate":
            src=rng.choice(nonempty)
            if not cand[src]: continue
            pos=rng.randrange(len(cand[src])); customer=cand[src].pop(pos)
            dst=rng.randrange(len(cand))
            insert_at=rng.randrange(len(cand[dst])+1)
            cand[dst].insert(insert_at,customer)
            cand=[r for r in cand if r]
        elif move=="swap":
            if len(nonempty)<2: continue
            a,b=rng.sample(nonempty,2)
            if not cand[a] or not cand[b]: continue
            ia,ib=rng.randrange(len(cand[a])),rng.randrange(len(cand[b]))
            ca,cb=cand[a][ia],cand[b][ib]
            if route_load(cand[a])-demands[ca]+demands[cb]>CAPACITY: continue
            if route_load(cand[b])-demands[cb]+demands[ca]>CAPACITY: continue
            cand[a][ia],cand[b][ib]=cb,ca
        else: # intra-route 2-opt
            a=rng.choice(nonempty)
            if len(cand[a])<3: continue
            i,j=sorted(rng.sample(range(len(cand[a])),2))
            if j-i<1: continue
            cand[a][i:j+1]=reversed(cand[a][i:j+1])
        if len(cand)<=K and all(route_load(r)<=CAPACITY for r in cand):
            check=validate_routes(cand,verbose=False)
            if check["feasible"] and solution_cost(cand)!=solution_cost(current):
                return cand
    return None

def simulated_annealing(start_routes, iterations=50000, seed=0, t0_factor=0.3, t_end=0.05):
    rng=random.Random(seed)
    current=[r[:] for r in start_routes]
    best=[r[:] for r in current]
    current_cost=solution_cost(current); best_cost=current_cost
    t0=max(1e-6,t0_factor*(current_cost/max(1,N_CUSTOMERS)))
    start=time.perf_counter()
    for step in range(iterations):
        fraction=step/max(1,iterations-1)
        temperature=t0*((t_end/t0)**fraction)
        candidate=make_neighbor(current,rng)
        if candidate is None: continue
        candidate_cost=solution_cost(candidate)
        delta=candidate_cost-current_cost
        if delta<=0 or rng.random()<math.exp(-delta/max(temperature,1e-12)):
            current,current_cost=candidate,candidate_cost
            if current_cost<best_cost:
                best,best_cost=[r[:] for r in current],current_cost
    elapsed=time.perf_counter()-start
    validation=validate_routes(best,verbose=False)
    return {"routes":best,"cost":best_cost,"runtime_s":elapsed,"feasible":validation["feasible"],
            "seed":seed,"iterations":iterations}

sa_result=simulated_annealing(initial_routes,iterations=50000,seed=0)
print({k:v for k,v in sa_result.items() if k!="routes"})
print("Initial cost:",solution_cost(initial_routes),"Improvement:",solution_cost(initial_routes)-sa_result["cost"])

## 4. Visual comparison

In [ ]:
def plot_routes(routes, title="CVRP routes"):
    plt.figure(figsize=(8,6))
    plt.scatter([coords[i][0] for i in range(1,n)],[coords[i][1] for i in range(1,n)],label="Customers")
    plt.scatter([coords[0][0]],[coords[0][1]],marker="s",s=120,label="Depot")
    for i in range(1,n): plt.annotate(str(i),(coords[i][0],coords[i][1]),fontsize=8)
    for route in routes:
        seq=[0]+route+[0]
        plt.plot([coords[i][0] for i in seq],[coords[i][1] for i in seq],marker="o",alpha=.8)
    plt.title(f"{title} | distance={solution_cost(routes)}")
    plt.xlabel("X"); plt.ylabel("Y"); plt.legend(); plt.grid(True); plt.show()

plot_routes(initial_routes,"Clarke–Wright")
plot_routes(sa_result["routes"],"Simulated Annealing")

**Important:** this is a teaching implementation with relocate, swap and 2-opt. The sample report describes additional Or-opt and 2-opt* moves and a 1,000,000-iteration budget. Add and test those only after this baseline is correct.